In [ ]:
from test_beta_package import *
%matplotlib inline

In [ ]:
# confirm the folder structure and load the variables
path_data_folder, path_raw_data_folder, path_process_data_folder, path_top_folder, path_info_dir, path_analysis_dir, \
path_plots_dir, path_single_pred_dir, path_multi_pred_dir = folder_structure()


In [ ]:
# load the processed pickle files  
data_continous_ratemaps, data_binned_glm = \
    processing_and_loading(path_raw_data_folder, path_process_data_folder, 
                           data_continous_ratemaps_pickle="rm_pre_data_diana_chasing2_notreheaded_XYZeuler_notricks_eo.pkl",
                           data_binned_glm_pickle="binned_4_GLM_rm_pre_data_diana_chasing2_notreheaded_XYZeuler_notricks_eo.pkl")


In [ ]:
# obtain the information from pickles
predictors_name_list, cells_id, animal_name = get_data_information(path_info_dir, data_continous_ratemaps)
print(predictors_name_list)

In [ ]:
# assign values to the inference parameters
dict_param = dict_parameters_hmm(path_info_dir, animal_name, num_dimen=1, num_categ_obs=2, N_iters=2,
                                 tolerance=10 ** -5, num_predictors=3, max_num_states=3,
                                 observation_type="input_driven_obs",
                                 transistion_type="inputdriven", optim_method="em")

In [ ]:
cells_index, tot_time = cells_selection_manual(path_info_dir, data_continous_ratemaps, data_binned_glm, dict_param, 'cells_selection.csv')

In [ ]:
glmhmms_ista, process_neur, inpts, T, tot_masked_indices, path_plots_list, dict_objects_multicov = \
   data_structure_multipredictor(path_analysis_dir, path_info_dir, path_multi_pred_dir, data_continous_ratemaps,
                            data_binned_glm, dict_param, tot_time, cells_index, best_predictors='best_predictors.csv')
print(glmhmms_ista)

In [ ]:
fit_ll_states_list, glmhmms_ista, time_states_comp = inference_section_multi_predictor(path_analysis_dir, path_info_dir, dict_param,
                                                                       glmhmms_ista=glmhmms_ista,
                                                                       process_neur=process_neur,
                                                                       inputs_list=inpts)

In [ ]:
log_like_evolution_per_states(path_analysis_dir, path_info_dir, fit_ll_states_list=fit_ll_states_list, dict_processed_objects=None,
                                  multipredictor=0)

In [ ]:
%%capture cap --no-stderr
posterior_probs_list = posterior_prob_process(path_info_dir, path_analysis_dir, dict_param=dict_param, glmhmms_ista=glmhmms_ista, process_neur=process_neur,
                           inputs_list=inpts, multipredictor=0)

In [ ]:
posterior_probs_list

In [ ]:
# %%capture cap --no-stderr
# take the cumulative occupancy per state
states_occupancies = states_occupancies_computation(path_analysis_dir, path_info_dir, posterior_probs_list=posterior_probs_list, multipredictor=0)


In [ ]:
print(states_occupancies)

In [ ]:
states_occupancies_histogram(path_analysis_dir, path_info_dir, dict_param=dict_param, states_occupancies=states_occupancies, multipredictor=0)

In [ ]:
transition_prob_matrix(path_analysis_dir, path_info_dir, dict_param=0, dict_processed_objects=0, comp_istance=1, multipredictor=0)

In [ ]:
weights_distribution_histogram(path_analysis_dir, path_info_dir, multipredictor=0)